In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from timm import create_model
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler


/home/sara/distilaimedical/distilvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()
token = os.getenv("HF_TOKEN")
if token:
    print(f"Loaded token.")
else:
    print("Token not found.")

Loaded token.


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [4]:
def find_fg_indices(mask, axis):
	is_fg = mask.any(axis=axis)
	fg_indices = np.flatnonzero(is_fg)
	if fg_indices.size == 0:
		return None
	return fg_indices[0], fg_indices[-1]

class CropBackground:
	def __init__(self, threshold=5):
		self.threshold = threshold

	def __call__(self, img):
		arr = np.array(img)
		r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
		mask = (r > self.threshold) | (g > self.threshold) | (b > self.threshold)
		h, w = mask.shape
		row_bounds = find_fg_indices(mask, axis=1)
		col_bounds = find_fg_indices(mask, axis=0)
		rmin, rmax = row_bounds if row_bounds is not None else (0, h - 1)
		cmin, cmax = col_bounds if col_bounds is not None else (0, w - 1)
		return img.crop((cmin, rmin, cmax + 1, rmax + 1))



"First, we cropped out all backgrounds and resized the images to 512 × 512 pixels. (...) In our experiments, we applied translation,stretching, rotation, flipping, and colour augmentation "

In [5]:
img_size = 32

transform_train = transforms.Compose([
    CropBackground(threshold=5),
    transforms.Resize((img_size, img_size)),
	transforms.RandomAffine(
		# translation,
        translate=(0.1, 0.1),
		# stretching,
		scale=(0.9, 1.1), 
		# rotation,
        degrees=15,
		
		fill=0
    ),
	# flipping, 
    transforms.RandomHorizontalFlip(),
	# and colour augmentation
    transforms.ColorJitter(brightness=0.5, contrast=1, saturation=0.1, hue=0.5),

    transforms.ToTensor(),
])

transform_eval = transforms.Compose([
	CropBackground(threshold=5),
	transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

In [6]:
class DDR_Dataset(Dataset):
    def __init__(self, root, mode, transform=None):
        self.img_dir = os.path.join(root, mode)
        self.transform = transform
        self.data = []

        labels_file = mode + ".txt"
        path = os.path.join(root, labels_file)
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                img_name, grade = line.split()
                self.data.append((img_name, int(grade)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_name, label = self.data[index]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        
        return img, label

In [7]:
root = "./ddr/DDR-dataset/DR_grading"
trainset = DDR_Dataset(root, "train", transform=transform_train)
valset = DDR_Dataset(root, "valid", transform=transform_eval)
testset = DDR_Dataset(root, "test", transform=transform_eval)

In [8]:
def get_dataloaders(seed=42):
    set_seed(seed)
    g = torch.Generator()
    g.manual_seed(seed)
    trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=4, worker_init_fn=seed_worker, generator=g, pin_memory=True)
    valloader = DataLoader(valset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    return trainloader,valloader, testloader

In [9]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

In [10]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

In [11]:
def execution_loop(model, optimizer, epochs, training_function, lr_scheduler, print_stats=True, **kwargs):
  trainloader, valloader, testloader = get_dataloaders(42)
  criterion = nn.CrossEntropyLoss()
  scaler = torch.amp.GradScaler('cuda')
  if print_stats:
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
    print("-" * 52)
  for epoch in range(epochs):
      current_lr = optimizer.param_groups[0]['lr']
      tr_loss, tr_acc = training_function(model, trainloader, optimizer, criterion, scaler, **kwargs)
      vl_loss, vl_acc = evaluate(model, valloader, criterion)
      if print_stats:
        print(f"{epoch+1:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}")
      lr_scheduler.step()
  test_loss, test_acc = evaluate(model, testloader, criterion)
  print(f"\nTest metrics:  {test_loss:>8.4f}  {test_acc:>7.4f}")

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device = ", device)

device =  cuda


In [ ]:
num_epochs = 5

set_seed(42)
import timm
convnext = create_model('convnext_tiny.fb_in22k', pretrained=True, num_classes=6)
convnext.to(device)
optim = torch.optim.Adam(convnext.parameters(), lr=1e-3)
scheduler = lr_scheduler.CosineAnnealingLR(optim, T_max=num_epochs, eta_min=1e-6)

execution_loop(convnext, optim, num_epochs, train_epoch, scheduler, print_stats=True)

Epoch  Train Loss  Train Acc  Val Loss  Val Acc
----------------------------------------------------
